<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Build Three Agent Tools (sol) </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

Exercise: Build Three Agent Tools

**Purpose**

Implement at least three tools for an LLM-powered agent:

*   One retrieval tool
*   One utility tool
*   One simulated external API tool

Then validate that the agent correctly selects the right tool for different test prompts.

**What You Will Learn**

1. Why agents need tools

2. How to create tools with the @tool decorator

3. Retrieval-Augmented Generation (RAG) basics

3. Utility functions vs. external API simulations

4. How an agent reasons and picks the correct tool (ReAct pattern)

5. Testing & debugging tool selection with real data

#1. Setup

In [1]:
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*"

In [2]:
!pip install sentence-transformers

In [3]:
# Install dependencies

import os
import getpass
import requests
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup done!")

Enter OpenAI API Key: ··········
Setup done!


#Task 1: Real Retrieval Tool (RAG)

In [5]:
docs = [
    Document(page_content="The Eiffel Tower, located in Paris, France, is 330 meters (1,083 ft) tall. It was built in 1889 and receives approximately 7 million visitors annually."),
    Document(page_content="The Burj Khalifa, located in Dubai, UAE, is the world's tallest building at 828 meters (2,717 ft). It was completed in 2010."),
    Document(page_content="OpenAI was founded in 2015 by Sam Altman, Elon Musk, and others. Its mission is to ensure that artificial general intelligence benefits all of humanity."),
    Document(page_content="Tesla, Inc. is an American multinational automotive and clean energy company headquartered in Austin, Texas. It was founded in 2003."),
    Document(page_content="The Colosseum, an oval amphitheatre in the centre of the city of Rome, Italy, was built in 70-80 AD. It could hold an estimated 50,000 to 80,000 spectators."),
    Document(page_content="Google, founded by Larry Page and Sergey Brin in 1998, is headquartered in Mountain View, California. It specializes in Internet-related services and products."),
    Document(page_content="Mount Everest, the Earth's highest mountain above sea level, is located in the Himalayas, between Nepal and Tibet. Its peak is 8,848.86 meters (29,031.7 ft) above sea level.")
]

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

@tool
def retrieve_knowledge(query: str) -> str:
    """Searches a real knowledge base containing facts about landmarks, companies, and history."""
    retrieved_docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in retrieved_docs])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Task 2: Utility Tool

In [7]:
@tool
def calculate(expression: str) -> str:
    """Performs mathematical calculations safely."""
    try:
        allowed_names = {"round": round, "abs": abs}
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Calculation error: {str(e)}"

#Task 3: Real External API Tool (Weather)

In [8]:
@tool
def get_weather(location: str) -> str:
    """Gets real current weather using wttr.in public service."""
    try:
        url = f"https://wttr.in/{location.replace(' ', '+')}?format=%C+%t+%w"
        response = requests.get(url, timeout=8)
        response.raise_for_status()
        return f"Weather in {location.title()}: {response.text.strip()}"
    except Exception as e:
        return f"Could not fetch weather for {location}. Error: {str(e)}"

#Create Agent

In [9]:
tools = [retrieve_knowledge, calculate, get_weather]

prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

print("Multi-tool agent is ready!")

Multi-tool agent is ready!


#Test Prompts (Validation)

In [10]:
test_prompts = [
    "What is the current weather in Paris right now?",
    "How tall is the Eiffel Tower and how many visitors does it get?",
    "Calculate 15% of 850 and then add 200",
    "Tell me about the Burj Khalifa",
    "What's the weather like in Tokyo today and what's 2**10?",
    "Who founded OpenAI and what is Tesla's headquarters city?"
]

for prompt in test_prompts:
    print(f"\n{'='*60}")
    print(f"QUERY: {prompt}")
    result = agent_executor.invoke({"input": prompt})
    print("ANSWER:", result["output"])


QUERY: What is the current weather in Paris right now?


> Entering new AgentExecutor chain...
I need to find the current weather in Paris. 
Action: get_weather
Action Input: "Paris"Weather in Paris: Partly cloudy +58Â°F â6mphI have retrieved the current weather information for Paris. It is partly cloudy with a temperature of 58°F and wind speed of 6 mph. 
Final Answer: The current weather in Paris is partly cloudy with a temperature of 58°F and wind speed of 6 mph.

> Finished chain.
ANSWER: The current weather in Paris is partly cloudy with a temperature of 58°F and wind speed of 6 mph.

QUERY: How tall is the Eiffel Tower and how many visitors does it get?


> Entering new AgentExecutor chain...
I need to find information about the height of the Eiffel Tower and its annual visitor statistics. I'll start by retrieving knowledge about the Eiffel Tower.  
Action: retrieve_knowledge  
Action Input: "Eiffel Tower height and visitor statistics"  The Eiffel Tower, located in Paris, Fran